In [ ]:
from google.colab import files
uploaded = files.upload()
uploaded = files.upload()

Saving al_east_schedule.csv to al_east_schedule (1).csv


Saving al_east_distance_matrix.csv to al_east_distance_matrix (1).csv


In [33]:
import pandas as pd
import numpy as np
import pyomo.environ as pyo
import shutil

# Load Datasets
df_div = pd.read_csv('al_east_schedule.csv')
dist_df = pd.read_csv('al_east_distance_matrix.csv', index_col=0)

# Extract Pyomo Sets
teams_list = list(df_div['Home'].unique())
stadiums_list = list(df_div['Location'].unique())
days_list = sorted(list(df_div['Date'].unique()))

# Build Parameter Dictionaries
dist_dict = dist_df.unstack().to_dict()
home_stadiums = df_div.groupby('Home')['Location'].agg(lambda x: x.mode()[0]).to_dict()

matchups = df_div.groupby(['Visitor', 'Home']).size().reset_index(name='games')
matchups_dict = {(row['Visitor'], row['Home']): row['games'] for index, row in matchups.iterrows()}

In [39]:
model = pyo.ConcreteModel()

# Sets & Parameters
model.T = pyo.Set(initialize=teams_list)
model.C = pyo.Set(initialize=stadiums_list)
model.D = pyo.Set(initialize=days_list)

model.Dist = pyo.Param(model.C, model.C, initialize=dist_dict)
model.H = pyo.Param(model.T, initialize=home_stadiums)
model.M = pyo.Param(model.T, model.T, initialize=matchups_dict, default=0)

# Decision Variables
model.A = pyo.Var(model.T, model.C, model.C, model.D, domain=pyo.Binary) # Travel
model.G = pyo.Var(model.T, model.T, model.D, domain=pyo.Binary)          # Matchups
model.L = pyo.Var(model.T, model.C, model.D, domain=pyo.Binary)          # Location

In [40]:
# Objective: Minimize Total Travel Distance
def obj_rule(m):
    return sum(m.Dist[b, c] * m.A[a, b, c, d] for a in m.T for b in m.C for c in m.C for d in m.D)
model.obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

In [41]:
# Constraints
def single_location_rule(m, a, d):
    return sum(m.L[a, c, d] for c in m.C) == 1
model.c_single_loc = pyo.Constraint(model.T, model.D, rule=single_location_rule)

def single_game_rule(m, a, d):
    away_games = sum(m.G[a, j, d] for j in m.T if j != a)
    home_games = sum(m.G[i, a, d] for i in m.T if i != a)
    return away_games + home_games <= 1
model.c_single_game = pyo.Constraint(model.T, model.D, rule=single_game_rule)

def required_matchups_rule(m, a, j):
    if a == j: return pyo.Constraint.Skip
    return sum(m.G[a, j, d] for d in m.D) == m.M[a, j]
model.c_req_matchups = pyo.Constraint(model.T, model.T, rule=required_matchups_rule)

def away_stadium_link_rule(m, a, j, d):
    if a == j: return pyo.Constraint.Skip
    return m.G[a, j, d] <= m.L[a, m.H[j], d]
model.c_away_link = pyo.Constraint(model.T, model.T, model.D, rule=away_stadium_link_rule)

def home_stadium_link_rule(m, a, j, d):
    if a == j: return pyo.Constraint.Skip
    return m.G[a, j, d] <= m.L[j, m.H[j], d]
model.c_home_link = pyo.Constraint(model.T, model.T, model.D, rule=home_stadium_link_rule)

def flow_arrival_rule(m, a, c, d):
    return m.L[a, c, d] == sum(m.A[a, b, c, d] for b in m.C)
model.c_flow_arr = pyo.Constraint(model.T, model.C, model.D, rule=flow_arrival_rule)

def flow_departure_rule(m, a, b, d):
    current_idx = days_list.index(d)
    if current_idx == 0: return pyo.Constraint.Skip
    prev_d = days_list[current_idx - 1]
    return m.L[a, b, prev_d] == sum(m.A[a, b, c, d] for c in m.C)
model.c_flow_dep = pyo.Constraint(model.T, model.C, model.D, rule=flow_departure_rule)

In [ ]:
# Configure & Run Solver
cbc_path = shutil.which('cbc')
solver = pyo.SolverFactory('cbc', executable=cbc_path)
solver.options['sec'] = 600
solver.options['ratio'] = 0.05

print("Optimizing schedule... (Maximum runtime: 10 minutes)")
results = solver.solve(model, tee=False)

# Extract Optimal Schedule
schedule_data = [
    {'Date': d, 'Team': a, 'Location': c}
    for d in model.D for a in model.T for c in model.C
    if pyo.value(model.L[a, c, d]) > 0.5
]
opt_df = pd.DataFrame(schedule_data).sort_values(by=['Team', 'Date'])

Optimizing schedule... (Maximum runtime: 10 minutes)


## Extracting The Optimized Schedule

In [ ]:
actual_total_distance = 0
for team in teams_list:
    team_games = df_div[(df_div['Home'] == team) | (df_div['Visitor'] == team)].sort_values('Date')
    locations = team_games['Location'].tolist()
    team_dist = sum(dist_df.loc[locations[i], locations[i+1]] for i in range(len(locations)-1))
    actual_total_distance += team_dist

optimized_distance = pyo.value(model.obj)
savings = actual_total_distance - optimized_distance



--- OPTIMIZED TRAVEL SCHEDULE ---
    Date Team Location
20250327  BAL    BAL12
20250327  BOS    BAL12
20250327  NYA    TOR02
20250327  TBA    TOR02
20250327  TOR    TAM02


In [ ]:
# Output Report
print("\n--- OPTIMIZATION REPORT ---")
print(f"Solver Status: {results.solver.status}")
print(f"Termination: {results.solver.termination_condition}")
print("-" * 25)
print(f"Actual 2025 Schedule: {actual_total_distance:,.1f} miles")
print(f"Optimized Schedule:   {optimized_distance:,.1f} miles")
print(f"Total Savings:        {savings:,.1f} miles ({(savings/actual_total_distance)*100:.1f}%)")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Distance Travel

In [ ]:
opt_df.to_csv('optimized_al_east_route.csv', index=False)


--- REAL-WORLD BASELINE COMPARISON ---
TOR Actual Travel: 7,131.6 miles
BAL Actual Travel: 6,392.2 miles
BOS Actual Travel: 6,481.2 miles
TBA Actual Travel: 10,561.8 miles
NYA Actual Travel: 5,208.1 miles
------------------------------
TOTAL ACTUAL DISTANCE:     35,774.9 miles
TOTAL OPTIMIZED DISTANCE:  12,745.2 miles

Optimization Savings: 23,029.8 miles (64.4%)
